# Stage 3b — Retrieval Demo (RAG)

Loads the index built by `03-build-index.ipynb` from `data/db/chroma/`
(same way the app does) and demonstrates the full grounded retrieval
pipeline — the same one the app runs at query time (`retriever.py`):

```
query
  ├──► Vector search (nomic-embed-text)
  └──► BM25 keyword search (chunk texts from Chroma)
        │
        ▼
  RRF fusion (0.5/0.5, k=60) → top-10 candidates
        │
        ▼
  LLM rerank (0–10) → keep top-3
        │
        ▼
  Refusal gate (top < 5.0 → "not in the offers")
        │
        ▼
  Grounded answer with inline [AG####] citations
```

The last cell runs a smoke test with four questions: a wording question,
a price question with an explicit offer reference (tests citations), a
question the corpus cannot answer (tests the refusal gate), and a
cross-offer pattern question ("which payment wording do we usually use
for offers above 3000 EUR?"). The fourth question exposes the limit of
standard RAG: retrieval finds the right chunks (high rerank score),
but the answer model refuses to generalize from top-k chunks — the
honest failure mode. The fix is the statistics route in
`05-retrieval-demo-full.ipynb` (full scan + code reduce).
> **This notebook: pure RAG** — one pipeline for every question.
> For the production-grade version with the app's router (deterministic
> price answers, clarification, aggregation limits), see
> `05-retrieval-demo-full.ipynb` (Stage 3c).


## Setup — Environment, LLM & Index

Load `.env`, create the OpenAI client (thinking OFF, temp 0),
open the Chroma collection, create the Ollama embedding model.

In [ ]:
import os, re, json, time
from pathlib import Path
from dotenv import load_dotenv

# Load .env (walk up: notebooks/ -> 01-submission/ -> final-project/)
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    dotenv_file = path / ".env"
    if dotenv_file.exists():
        load_dotenv(dotenv_file, override=True)
        print(f"✅ Loaded .env from {dotenv_file}")
        break

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "not-needed")
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

DEMO_DIR = Path.cwd().parent            # 01-submission
CHROMA_DIR = DEMO_DIR / "data" / "db" / "chroma"
COLLECTION = "offers"

# Retrieval pipeline parameters (same defaults as the app's config.py)
RRF_K = 60
W_VEC, W_BM25 = 0.5, 0.5
RERANK_TOP_N = 10
KEEP = 3
REFUSAL_THRESHOLD = 5.0

# --- LLM (remote vLLM, OpenAI-compatible; thinking OFF, temperature 0) ---
from openai import OpenAI
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def llm_chat(prompt: str) -> str:
    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=4096,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content

# --- Index (loaded from disk, exactly like the app) ---
import chromadb
from llama_index.embeddings.ollama import OllamaEmbedding
from types import SimpleNamespace

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION)
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)
print(f"✅ Index: {collection.count()} chunks in '{COLLECTION}' ({CHROMA_DIR})")
print(f"✅ LLM: {LLM_MODEL} at {LLM_BASE_URL} (thinking off, temp 0)")
print(f"✅ Pipeline: RRF k={RRF_K} w={W_VEC}/{W_BM25} → rerank top-{RERANK_TOP_N} → keep {KEEP} → refusal < {REFUSAL_THRESHOLD}")

## Step 1: Hybrid Search (BM25 + Vector + RRF)

Build the BM25 corpus from Chroma chunk texts. Two search arms
(vector via nomic-embed-text, BM25 keyword) fused with weighted RRF
(0.5/0.5, k=60) → top-10 candidates.

In [ ]:
from rank_bm25 import BM25Okapi
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
from nltk.tokenize import word_tokenize

# --- BM25 corpus: the chunk texts straight from Chroma (no separate artifact) ---
_chroma_res = collection.get(include=["documents", "metadatas"])
node_list = [
    SimpleNamespace(node_id=_id, text=_doc, metadata=dict(_meta or {}))
    for _id, _doc, _meta in zip(
        _chroma_res["ids"], _chroma_res["documents"], _chroma_res["metadatas"]
    )
]
corpus_tokens = [word_tokenize(n.text.lower()) for n in node_list]
bm25 = BM25Okapi(corpus_tokens)
print(f"🔤 BM25 corpus: {len(node_list)} chunks from Chroma")

def vector_search(query, top_n=10, angebot_id=None):
    """Vector search via the same embedding model the index was built with."""
    q_emb = embed_model.get_text_embedding(query)
    where = {"angebot_id": angebot_id} if angebot_id else None
    res = collection.query(query_embeddings=[q_emb], n_results=top_n,
                           where=where,
                           include=["documents", "metadatas", "distances"])
    out = []
    for _id, _doc, _meta, _dist in zip(res["ids"][0], res["documents"][0],
                                       res["metadatas"][0], res["distances"][0]):
        meta = dict(_meta or {})
        meta["vec_score"] = max(0.0, 1.0 - _dist / 2.0)   # L2 -> similarity
        out.append(SimpleNamespace(node_id=_id, text=_doc, metadata=meta))
    return out

def bm25_search(query, top_n=10, angebot_id=None):
    """Keyword search over the indexed chunks (optionally restricted to one offer)."""
    scores = bm25.get_scores(word_tokenize(query.lower()))
    if angebot_id:
        for i, n in enumerate(node_list):
            if n.metadata.get("angebot_id") != angebot_id:
                scores[i] = -1.0
    top_ids = scores.argsort()[::-1][:top_n]
    return [node_list[i] for i in top_ids]

def rrf_fuse(vec_results, bm25_results, w_vec=W_VEC, w_bm25=W_BM25, k=RRF_K, top_n=10):
    """Merge two ranked node lists via weighted Reciprocal Rank Fusion."""
    scores, node_by_id = {}, {}
    for rank, node in enumerate(vec_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_vec / (k + rank + 1)
        node_by_id[node.node_id] = node
    for rank, node in enumerate(bm25_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_bm25 / (k + rank + 1)
        node_by_id[node.node_id] = node
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_n]
    out = []
    for nid in ranked:
        node = node_by_id[nid]
        node.metadata["rrf_score"] = scores[nid]
        out.append(node)
    return out

def hybrid_search(query, top_n=RERANK_TOP_N, angebot_id=None):
    """Vector + BM25, merged with RRF. Returns top_n candidates.

    If `angebot_id` is given (the question names a specific offer), both
    search paths are restricted to that offer — same as the app's query.py.
    """
    vec = vector_search(query, top_n=top_n, angebot_id=angebot_id)
    kw = bm25_search(query, top_n=top_n, angebot_id=angebot_id)
    return rrf_fuse(vec, kw, top_n=top_n)

print("✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5)")

## Step 2: Rerank, Refusal Gate & Cited Answer

LLM rerank (0–10) → refusal gate (< 5.0) → grounded answer with inline
`[AG#### | S. X]` citations (verbatim quotes + deterministic page lookup).

In [ ]:
def _strip_think(text):
    """Remove  blocks (Qwen thinking mode) before parsing."""
    return re.sub(r"\x3cthink.*?\x3c/endthink\x3e", "", text, flags=re.DOTALL).strip()

PAGE_RE = re.compile(r"\[Seite (\d+) von \d+\]")

def _norm(s):
    """Normalize whitespace + quote variants for robust quote matching."""
    s = re.sub(r"\s+", " ", s)
    for a, b in [("\u201e", '"'), ("\u201c", '"'), ("\u201d", '"'),
                 ("\u201a", "'"), ("\u2018", "'"), ("\u2019", "'"),
                 ("\u2026", "...")]:
        s = s.replace(a, b)
    return s.strip()

def page_of_quote(chunk_text, quote):
    """Deterministic page lookup: find the verbatim quote in the chunk text and
    return the number of the LAST '[Seite X von Y]' marker before it.
    Returns None if the quote cannot be located (never invent a page)."""
    nq, nt = _norm(quote), _norm(chunk_text)
    pos = nt.find(nq)
    if pos < 0:
        return None  # quote not verbatim in chunk -> no page, never guess
    pages = [int(m.group(1)) for m in PAGE_RE.finditer(nt) if m.start() < pos]
    return pages[-1] if pages else 1

def citation_label(node, quote=None):
    """Citation: [AG#### | S. X] — page resolved deterministically from the
    page markers that are already inside the chunk text. Falls back to
    [AG####] when the quote cannot be located."""
    oid = node.metadata.get("angebot_id", "unknown offer")
    page = page_of_quote(node.text, quote) if quote else None
    return f"[{oid} | S. {page}]" if page else f"[{oid}]"

def llm_rerank(query, candidates, keep=KEEP):
    """Score hybrid candidates with the LLM (0-10) and return the top `keep`."""
    snippets = []
    for i, node in enumerate(candidates, 1):
        snippets.append(f"[{i}] ({citation_label(node)})\n{node.text[:1200]}")
    numbered = "\n\n".join(snippets)

    prompt = (
        "You are a search reranker. Given a query and numbered text passages, "
        "score each passage 0-10 for how well it ANSWERS the query.\n"
        "10 = directly and fully answers, 5 = partially related, 0 = irrelevant.\n"
        "Return ONLY a JSON object mapping passage number to score, e.g. {\"1\": 8, \"2\": 3}.\n\n"
        f"QUERY: {query}\n\nPASSAGES:\n{numbered}"
    )
    # Retry loop: the remote LLM occasionally returns an empty/malformed
    # completion. Retry up to 3x with a short backoff.
    t0 = time.time()
    scores = {}
    for attempt in range(3):
        text = llm_chat(prompt)
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                scores = json.loads(m.group(0))
                break
            except json.JSONDecodeError:
                scores = {}
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    elapsed = time.time() - t0

    scored = []
    for i, node in enumerate(candidates, 1):
        try:
            s = float(scores.get(str(i), 0))
        except (ValueError, TypeError):
            s = 0.0
        node.metadata["rerank_score"] = s
        scored.append(node)
    scored.sort(key=lambda n: n.metadata["rerank_score"], reverse=True)
    print(f"   rerank: {elapsed:.1f}s over {len(candidates)} candidates")
    return scored[:keep]

AG_RE = re.compile(r"\bAG\d{4}\b")

def answer(query, top_n=RERANK_TOP_N, keep=KEEP):
    """Full grounded pipeline: hybrid -> rerank -> (refuse if low) -> cited answer.

    If the question names a specific offer (AG####), retrieval is restricted
    to that offer via a metadata filter — same as the app's query.py.
    Returns (answer_text, ranked_nodes, top_score).
    """
    m = AG_RE.search(query)
    angebot_id = m.group(0) if m else None
    ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n, angebot_id=angebot_id), keep=keep)
    top_score = ranked[0].metadata.get("rerank_score", 0.0)

    if top_score < REFUSAL_THRESHOLD:
        msg = (f"Die Angebote enthalten dazu keine verlässliche Antwort "
               f"(bester Kandidat: {top_score:.0f}/10).")
        return msg, ranked, top_score

    context = [f"[{i}] ({citation_label(node)})\n{node.text}"
               for i, node in enumerate(ranked, 1)]
    numbered_context = "\n\n".join(context)

    prompt = (
        "Du bist ein präziser RAG-Assistent für Angebote eines Filmstudios. "
        "Beantworte AUSSCHLIESSLICH anhand der nummerierten Kontexte.\n"
        "Regeln:\n"
        "1. Die Antwort kann in MEHREREN Chunks liegen. Gliedere nach Teilaspekt "
        "(ein kurzer Absatz oder Bullet pro Aspekt).\n"
        "2. Zitiere NACH JEDEM Aspekt inline die Quelle in eckigen Klammern mit "
        "der Angebots-ID — z. B. '...Zahlung innerhalb von 14 Tagen [AG1001].' "
        "Keine Zitatliste am Ende.\n"
        "3. Belege WÖRTLICH: Führe die entscheidende Passage aus dem Kontext "
        "exakt so an, wie sie dort steht, in deutschen Anführungszeichen "
        "(\u201e...\u201c).\n"
        "4. Ordne jede Aussage dem Chunk zu, der sie tatsächlich enthält.\n"
        "5. Kein externes Wissen. Was nicht abgedeckt ist, sagst du: 'Nicht in den Angeboten enthalten.'\n"
        "6. Antworte auf Deutsch.\n\n"
        f"KONTEXT:\n{numbered_context}\n\n"
        f"FRAGE: {query}\n\nANTWORT:"
    )
    answer_text = _strip_think(llm_chat(prompt)).strip()
    answer_text = upgrade_citations(answer_text, ranked)
    return answer_text, ranked, top_score

QUOTE_RE = re.compile(r"\u201e(.+?)\u201c", flags=re.DOTALL)
CITE_RE = re.compile(r"\[(AG\d{4})(?:\s*\|[^\]]*)?\]")

def upgrade_citations(answer_text, ranked):
    """Post-processing: for every [AG####] citation, find the verbatim quote
    that belongs to it in that offer's chunks and append the page:
    [AG####] -> [AG#### | S. X]. Deterministic — no LLM involved.
    Citations that cannot be resolved are left as [AG####]."""
    quotes = [m.group(1).strip() for m in QUOTE_RE.finditer(answer_text)]
    if not quotes:
        return answer_text
    # map offer id -> chunk texts (only the kept, high-scoring nodes)
    texts = {}
    for node in ranked:
        texts.setdefault(node.metadata.get("angebot_id"), []).append(node.text)
    # pair each citation with the nearest preceding quote
    pairs = []  # (cite_match, quote)
    qpos = [(m.start(), m.group(1).strip()) for m in QUOTE_RE.finditer(answer_text)]
    for m in CITE_RE.finditer(answer_text):
        preceding = [q for p, q in qpos if p < m.start()]
        if not preceding:
            continue
        quote = preceding[-1]
        oid = m.group(1)
        page = None
        for t in texts.get(oid, []):
            page = page_of_quote(t, quote)
            if page:
                break
        if page:
            pairs.append((m, f"[{oid} | S. {page}]"))
    # replace from the end so positions stay valid
    for m, repl in reversed(pairs):
        answer_text = answer_text[:m.start()] + repl + answer_text[m.end():]
    return answer_text


## Step 3: Smoke Test (4 questions)

One question per capability: wording, price+ID (citations), refusal gate,
and the cross-offer pattern question that exposes the top-k limit.

In [ ]:
SMOKE = [
    "Wie sind die Zahlungsbedingungen formuliert?",
    "Wie hoch war der Preis für Color Grading in AG1001?",
    "Wie hoch war der Studio-Umsatz 2019?",   # not in the offers -> must refuse
    # 4) wording/pattern question -> standard RAG across offers
    "Hey, welche Zahlungsformulierung nutzen wir in der Regel bei Angeboten "
    "über 3000 Euro?",
]

for q in SMOKE:
    print("=" * 70)
    print("Q:", q)
    print("-" * 70)
    # Mirror answer(): if the question names an offer, restrict retrieval to it.
    m = AG_RE.search(q)
    oid = m.group(0) if m else None
    if oid:
        print(f"(offer reference detected -> retrieval filtered to {oid})")
    cands = hybrid_search(q, top_n=RERANK_TOP_N, angebot_id=oid)
    print("Hybrid candidates (top 5):")
    for n in cands[:5]:
        md = n.metadata
        print(f"  {md.get('angebot_id')} | {md.get('datum', '—'):<12} | "
              f"preis={md.get('preis', '—'):<10} | rrf={md['rrf_score']:.4f}")
    a, ranked, top = answer(q)
    print(f"Top rerank score: {top:.0f}/10")
    print()
    print(a)
    print()